# Bangla Dialect Dataset — Data Preprocessing Pipeline

This notebook consolidates the full preprocessing pipeline for the Bangla dialect translation dataset.
It covers the following steps in order:

1. **Dataset Collection & Merging** — Load multiple source datasets and unify them into per-dialect CSVs
2. **Deduplication** — Remove duplicate dialect sentences, retaining rows with English translations where possible
3. **Leakage Detection & Removal** — Ensure no overlap between train, validation, and test splits
4. **English Translation** — Use OpenAI GPT-4o-mini to translate missing Standard Bangla sentences
5. **Fill Missing Translations** — Match and fill missing English translations using the generated translations, then finalize clean datasets

**Dialects covered:** Barisal, Chittagong, Mymensingh, Noakhali, Sylhet

**Source datasets:** Vashantor, ONUBAD, ChatgaiyyaAlap, BIDWESH, BanglaRegionalTextCorpus, ANUBHUTI, ANCHOLIK-NER

---
## 0. Install Dependencies

In [ ]:
!pip install --quiet pandas numpy openai tqdm

In [ ]:
import os
import re
import time
import unicodedata

import pandas as pd
import numpy as np
from tqdm import tqdm
import openai

pd.set_option("display.max_colwidth", 200)

---
## 1. Dataset Collection & Merging

Load all source datasets and unify them into a standard schema:
`Standard_Bangla` | `dialect_speech` | `English_translation`

In [ ]:
# ── Load all source datasets ────────────────────────────────────────────────

# Vashantor (one file per dialect)
vashantor_barisal    = pd.read_csv("/content/drive/MyDrive/datasets/Vashantor/Vashantor_CSV_Format/Train/Barishal Train Translation.csv")
vashantor_chittagong = pd.read_csv("/content/drive/MyDrive/datasets/Vashantor/Vashantor_CSV_Format/Train/Chittagong Train Translation.csv")
vashantor_mymensingh = pd.read_csv("/content/drive/MyDrive/datasets/Vashantor/Vashantor_CSV_Format/Train/Mymensingh Train Translation.csv")
vashantor_noakhali   = pd.read_csv("/content/drive/MyDrive/datasets/Vashantor/Vashantor_CSV_Format/Train/Noakhali Train Translation.csv")
vashantor_sylhet     = pd.read_csv("/content/drive/MyDrive/datasets/Vashantor/Vashantor_CSV_Format/Train/Sylhet Train Translation.csv")

# Other sources
onubad      = pd.read_csv("/content/drive/MyDrive/datasets/ONUBAD/Sentence.csv")
chatgayyiya = pd.read_csv("/content/drive/MyDrive/datasets/ChatgaiyyaAlap A Dataset for Conversion from Chittagonian Dialect to Standard Bangla/Dataset_Chittagong_2.0.csv")
bidwesh     = pd.read_csv("/content/drive/MyDrive/datasets/BIDWESH A Bangla Regional Based Hate Speech Detect/Regional Translated Texts.csv")
brtc        = pd.read_csv("/content/drive/MyDrive/datasets/BanglaRegionalTextCorpus/Regional_cleaned_dataset-ardvs5 - Sheet1.csv")
anubhuti    = pd.read_csv("/content/drive/MyDrive/datasets/ANUBHUTI A COMPREHENSIVE CORPUS FOR SENTIMENT  ANA/ANUBHUTI_Translated_Texts.csv")
ancholich   = pd.read_csv("/content/drive/MyDrive/datasets/ANCHOLIK-NER/Regional_NER (Raw Sentences).csv")

In [ ]:
# ── Helper: build a unified 3-column DataFrame ───────────────────────────────

def make_df(std_bn=None, dialect=None, en=None):
    """Create a unified DataFrame with standard schema."""
    return pd.DataFrame({
        "Standard_Bangla":     std_bn,
        "dialect_speech":      dialect,
        "English_translation": en
    })

In [ ]:
# ── Accumulate per-dialect DataFrames ────────────────────────────────────────

dialects = {"Barisal": [], "Chittagong": [], "Noakhali": [], "Sylhet": [], "Mymensingh": []}

# ONUBAD
for d, col in {"Barisal": "Barisal Language", "Chittagong": "Chittagong Language", "Sylhet": "Sylhet Language"}.items():
    dialects[d].append(make_df(
        std_bn=onubad["Standard Bangla Lanuguage"],
        dialect=onubad[col],
        en=onubad["English Translation"]
    ))

# ChatgaiyyaAlap (Chittagong only)
dialects["Chittagong"].append(make_df(
    std_bn=chatgayyiya["\u09ac\u09be\u0982\u09b2\u09be"],
    dialect=chatgayyiya["\u099a\u099f\u09cd\u099f\u0997\u09cd\u09b0\u09be\u09ae"],
    en=None
))

# BIDWESH
for d, col in {"Chittagong": "Chittagong", "Noakhali": "Noakhali", "Barisal": "Barishal"}.items():
    dialects[d].append(make_df(
        std_bn=bidwesh["Standard Bangla"],
        dialect=bidwesh[col],
        en=None
    ))

# BanglaRegionalTextCorpus
for d in dialects:
    subset = brtc[brtc[" Labels"].str.lower() == d.lower()]
    if len(subset):
        dialects[d].append(make_df(
            std_bn=None,
            dialect=subset["Sentence"],
            en=subset["English Translation"]
        ))

# ANUBHUTI
for d, col in {"Chittagong": "Chittagong", "Noakhali": "Noakhali", "Sylhet": "Sylhet", "Mymensingh": "Mymenshingh"}.items():
    dialects[d].append(make_df(
        std_bn=anubhuti["Standard Bangla"],
        dialect=anubhuti[col],
        en=None
    ))

# ANCHOLIK-NER
for d, col in {"Barisal": "Barishal", "Chittagong": "Chittagong", "Noakhali": "Noakhali", "Sylhet": "Sylhet", "Mymensingh": "Mymensingh"}.items():
    dialects[d].append(make_df(
        std_bn=ancholich["Standard_Bangla"],
        dialect=ancholich[col],
        en=None
    ))

# Vashantor
vashantor_map = [
    ("Barisal",    vashantor_barisal,    "barishal_bangla_speech "),
    ("Chittagong", vashantor_chittagong, "chittagong_bangla_speech "),
    ("Mymensingh", vashantor_mymensingh, "mymensingh_bangla_speech "),
    ("Noakhali",   vashantor_noakhali,   "noakhali_bangla_speech "),
    ("Sylhet",     vashantor_sylhet,     "sylhet_bangla_speech "),
]
for d, src, dialect_col in vashantor_map:
    dialects[d].append(make_df(
        std_bn=src["bangla_speech "],
        dialect=src[dialect_col],
        en=src["english_speech"]
    ))

In [ ]:
# ── Concatenate and save raw merged dialect files ────────────────────────────

os.makedirs("/content/output/raw", exist_ok=True)

for d, dfs in dialects.items():
    final_df = pd.concat(dfs, ignore_index=True)
    final_df = final_df.dropna(subset=["dialect_speech"])
    final_df.to_csv(f"/content/output/raw/{d.lower()}_dialect.csv", index=False)
    print(f"Saved {d}: {len(final_df)} rows")

---
## 2. Deduplication

Remove duplicate `dialect_speech` entries within each dialect file.
When duplicates exist, the copy that has an English translation is kept.

In [ ]:
def deduplicate_keep_english(df):
    """Deduplicate on dialect_speech, keeping rows that have an English translation."""
    df = df.copy()
    df["has_english"] = (
        df["English_translation"].notna() &
        (df["English_translation"].astype(str).str.strip() != "")
    )
    # Sort so rows WITH English come first, then keep the first occurrence
    df = df.sort_values(by=["dialect_speech", "has_english"], ascending=[True, False])
    kept    = df[~df.duplicated(subset=["dialect_speech"], keep="first")].drop(columns="has_english")
    removed = df[ df.duplicated(subset=["dialect_speech"], keep="first")].drop(columns="has_english")
    return kept.reset_index(drop=True), removed.reset_index(drop=True)

In [ ]:
os.makedirs("/content/output/deduplicated", exist_ok=True)

dialect_names = ["barisal", "chittagong", "mymensingh", "noakhali", "sylhet"]

for d in dialect_names:
    df = pd.read_csv(f"/content/output/raw/{d}_dialect.csv")
    cleaned, removed = deduplicate_keep_english(df)
    cleaned.to_csv(f"/content/output/deduplicated/{d}_dialect.csv", index=False)
    print(f"{d.capitalize()}: {len(df)} → {len(cleaned)} rows  ({len(removed)} duplicates removed)")

---
## 3. Leakage Detection & Removal

Check for overlapping sentences between the training split (deduplicated above)
and the validation / test splits (from the Vashantor dataset).
Overlapping rows are removed from the validation and test sets.

In [ ]:
# Paths — update these to match your Google Drive layout
TRAIN_PATH = "/content/output/deduplicated/{}_dialect.csv"
VAL_PATH   = "/content/output/validation/{}_validation.csv"
TEST_PATH  = "/content/output/test/{}_test.csv"

In [ ]:
def get_dialect_col(dialect):
    """Return the dialect speech column name used in Vashantor val/test files."""
    mapping = {
        "barisal":    "barishal_bangla_speech ",
        "chittagong": "chittagong_bangla_speech ",
        "mymensingh": "mymensingh_bangla_speech ",
        "noakhali":   "noakhali_bangla_speech ",
        "sylhet":     "sylhet_bangla_speech ",
    }
    return mapping[dialect]


def find_overlap(train_df, other_df, other_std_col, other_dialect_col):
    """Return rows from other_df that appear in train_df."""
    return pd.merge(
        train_df[["Standard_Bangla", "dialect_speech"]],
        other_df[[other_std_col, other_dialect_col]],
        left_on=["Standard_Bangla", "dialect_speech"],
        right_on=[other_std_col, other_dialect_col],
        how="inner"
    )

In [ ]:
os.makedirs("/content/output/removed_duplicates", exist_ok=True)
STD_COL = "bangla_speech "

for d in dialect_names:
    dialect_col = get_dialect_col(d)

    train_df = pd.read_csv(TRAIN_PATH.format(d))
    val_df   = pd.read_csv(VAL_PATH.format(d))
    test_df  = pd.read_csv(TEST_PATH.format(d))

    # Strip whitespace for reliable matching
    for frame, cols in [
        (train_df, ["Standard_Bangla", "dialect_speech"]),
        (val_df,   [STD_COL, dialect_col]),
        (test_df,  [STD_COL, dialect_col]),
    ]:
        for c in cols:
            frame[c] = frame[c].astype(str).str.strip()

    tv_overlap = find_overlap(train_df, val_df,  STD_COL, dialect_col)
    tt_overlap = find_overlap(train_df, test_df, STD_COL, dialect_col)

    # Remove overlapping rows from val / test
    if not tv_overlap.empty:
        mask = val_df.set_index([STD_COL, dialect_col]).index.isin(
            tv_overlap.set_index(["Standard_Bangla", "dialect_speech"]).index
        )
        val_df[mask].to_csv(f"/content/output/removed_duplicates/{d}_val_removed.csv", index=False)
        val_df = val_df[~mask]

    if not tt_overlap.empty:
        mask = test_df.set_index([STD_COL, dialect_col]).index.isin(
            tt_overlap.set_index(["Standard_Bangla", "dialect_speech"]).index
        )
        test_df[mask].to_csv(f"/content/output/removed_duplicates/{d}_test_removed.csv", index=False)
        test_df = test_df[~mask]

    # Overwrite splits with clean versions
    train_df.to_csv(TRAIN_PATH.format(d), index=False)
    val_df.to_csv(VAL_PATH.format(d),     index=False)
    test_df.to_csv(TEST_PATH.format(d),   index=False)

    print(f"{d.capitalize()}: val leakage={len(tv_overlap)}, test leakage={len(tt_overlap)}")

---
## 4. English Translation via OpenAI GPT-4o-mini

Translate Standard Bangla sentences that have no English translation.
Sentences are sent in batches of 50 to reduce API call overhead.

In [ ]:
openai.api_key = ""  # ← paste your OpenAI API key here

In [ ]:
def translate_batch(sentences: list[str]) -> list[str]:
    """Translate a batch of Bangla sentences to English via GPT-4o-mini."""
    cleaned = [s if pd.notna(s) else "" for s in sentences]
    prompt = "Translate the following Bangla sentences to English, keeping numbering:\n"
    for i, s in enumerate(cleaned, start=1):
        prompt += f"{i}. {s}\n"

    try:
        response = openai.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a helpful translator from Bangla to English."},
                {"role": "user",   "content": prompt}
            ],
            temperature=0
        )
        lines = response.choices[0].message.content.strip().split("\n")
        translations = []
        for line in lines:
            translations.append(line.split(".", 1)[1].strip() if "." in line else line.strip())
        return translations[:len(sentences)]
    except Exception as e:
        print(f"Translation error: {e}")
        return [""] * len(sentences)

In [ ]:
# Load the Standard Bangla sentences that need translation
# (Replace the path below with the file that contains your untranslated sentences)
to_translate_df = pd.read_csv("/content/drive/MyDrive/merged_standard_bangla_english.csv")

# Only translate rows that are currently missing an English translation
missing_mask = (
    to_translate_df["English_translation"].isna() |
    (to_translate_df["English_translation"].astype(str).str.strip() == "") |
    (to_translate_df["English_translation"].astype(str).str.lower() == "nan")
)
sentences_to_translate = to_translate_df.loc[missing_mask, "Standard_Bangla"].tolist()
print(f"Sentences to translate: {len(sentences_to_translate)}")

In [ ]:
BATCH_SIZE = 50
all_translations = []

for start in tqdm(range(0, len(sentences_to_translate), BATCH_SIZE)):
    batch = sentences_to_translate[start : start + BATCH_SIZE]
    all_translations.extend(translate_batch(batch))
    time.sleep(1)  # Respect rate limits

# Write translations back to the DataFrame
to_translate_df.loc[missing_mask, "English_translation"] = all_translations
to_translate_df.to_csv("/content/translated_results.csv", index=False)
print("Saved translated_results.csv")

---
## 5. Fill Missing Translations & Finalize Datasets

Use the generated translations to fill `English_translation` gaps in each
dialect CSV. Unicode normalization is applied before matching to handle
Bangla encoding inconsistencies. Rows that remain unfilled are dropped.

In [ ]:
def normalize_bangla(text: str) -> str:
    """Apply Unicode NFC normalization and remove zero-width characters."""
    if pd.isna(text):
        return text
    text = unicodedata.normalize("NFC", str(text))
    text = re.sub(r"[\u200c\u200d]", "", text)   # remove zero-width joiners
    text = re.sub(r"\s+", " ", text)              # collapse whitespace
    return text.strip()


def is_missing_english(series: pd.Series) -> pd.Series:
    """Return a boolean mask for rows with a missing English translation."""
    return (
        series.isna() |
        (series.astype(str).str.strip() == "") |
        (series.astype(str).str.lower() == "nan")
    )


def fill_missing_english(df: pd.DataFrame, translation_dict: dict) -> pd.DataFrame:
    """Fill missing English translations using a normalized Bangla → English lookup."""
    df = df.copy()
    df["_norm"] = df["Standard_Bangla"].apply(normalize_bangla)
    mask = is_missing_english(df["English_translation"])
    df.loc[mask, "English_translation"] = df.loc[mask, "_norm"].map(translation_dict)
    df.drop(columns="_norm", inplace=True)
    return df

In [ ]:
# ── Build lookup dictionary from translated_results.csv ───────────────────────

translated = pd.read_csv("/content/translated_results.csv")
translated["Standard_Bangla"] = translated["Standard_Bangla"].astype(str).str.strip()
translated["English_translation"] = translated["English_translation"].astype(str).str.strip()

# Keep only rows with valid translations
translated = translated[~is_missing_english(translated["English_translation"])]

# Normalise keys
translated["_norm"] = translated["Standard_Bangla"].apply(normalize_bangla)
translation_dict = (
    translated
    .groupby("_norm")["English_translation"]
    .first()
    .to_dict()
)
print(f"Translation lookup entries: {len(translation_dict)}")

In [ ]:
# ── Apply to each dialect, drop rows still missing English, save final files ──

os.makedirs("/content/output/final", exist_ok=True)

for d in dialect_names:
    df = pd.read_csv(f"/content/output/deduplicated/{d}_dialect.csv")

    df = fill_missing_english(df, translation_dict)

    # Drop rows that still have no English translation
    before = len(df)
    df = df[~is_missing_english(df["English_translation"])].reset_index(drop=True)
    after = len(df)

    # Drop the temporary normalisation column if it lingered
    df = df[["Standard_Bangla", "dialect_speech", "English_translation"]]

    df.to_csv(f"/content/output/final/{d}_dialect_final.csv", index=False)
    print(f"{d.capitalize()}: {before} → {after} rows  ({before - after} dropped)")

---
## 6. Final Summary

Print row counts for all final dialect files.

In [ ]:
print("=" * 45)
print(f"{'Dialect':<15} {'Final Rows':>12}")
print("=" * 45)
for d in dialect_names:
    df = pd.read_csv(f"/content/output/final/{d}_dialect_final.csv")
    print(f"{d.capitalize():<15} {len(df):>12,}")
print("=" * 45)